# Dump the FINK schema on JSON file

Currrent: 
PORTAL : "https://api.lsst.fink-portal.org"
API version : "/api/v1/"

Dump for : "statistics", "tags", "conesearch", "objects"
            "sources", "fp", "cutouts"

Save a Json file with naming convention all_schema_"API version"_YYYYMMDD

In [ ]:
import requests
import pandas as pd
pd.set_option("display.max_rows", None)
from pprint import pprint

In [ ]:
FINK_API = "https://api.lsst.fink-portal.org"
API_V = "/api/v1/"

In [ ]:
api_version = API_V.replace("/", "_").strip("_")
print(api_version)  # Affiche : api_v1

In [ ]:
import os
import glob
from datetime import datetime

# 1. Chemin du dossier où se trouvent tes fichiers (modifie selon ton cas)
dossier = "./"  # Par défaut : dossier courant

# 2. Liste tous les fichiers correspondant au pattern
fichiers = glob.glob(os.path.join(dossier, f"all_schema_{api_version}_*.json"))

# 3. Si aucun fichier trouvé, affiche un message
FILE_OK = False
if not fichiers:
    print(f"Aucun fichier all_schema_{api_version}_*.json trouvé dans le dossier.")
else:
    # 4. Extrait la date de chaque fichier et trie par date
    fichiers_par_date = []
    for fichier in fichiers:
        # Extrait la date du nom de fichier (ex: 20260403)
        date_str = fichier.split(f"_{api_version}_")[1].split(".json")[0]
        date = datetime.strptime(date_str, "%Y%m%d")
        fichiers_par_date.append((date, fichier))

    # 5. Trie par date décroissante (le plus récent en premier)
    fichiers_par_date.sort(reverse=True, key=lambda x: x[0])

    # 6. Récupère le chemin du fichier le plus récent
    dernier_fichier = fichiers_par_date[0][1]

    # 7. Charge le fichier JSON
    import json
    with open(dernier_fichier, "r") as f:
        all_schema = json.load(f)

    # 8. Affiche le résultat
    print(f"Fichier chargé : {dernier_fichier}")
    FILE_OK = True

In [ ]:
def result_from_post(url_schema, endpoint):
    """ """
    payload = {"endpoint": f"{endpoint}"}
    res_schema = requests.post(url_schema, json=payload)
    res_schema.raise_for_status()
    schema = res_schema.json()
    return pd.DataFrame(schema)
    
def schema_to_dict(struc, df):
    result = {}

    for field, row in df.iterrows():
        for col, val in row.items():
            
            if isinstance(val, dict):
                prefix = col.strip()  # "r:", "f:", etc.
                prefix = prefix[prefix.index('(')+1 : prefix.index(')')]
                doc = val.get("doc", "").strip()
                key = f"{struc}/{prefix}{field}"
                result[key] = doc

    return result
#not used
def result_from_get(url):
    """ """
    response = requests.get(url)
    if response.status_code == 200:
        d = response.json()
        pprint(f"Nb items : {len(d)}")
        pprint("List of items :")
        df = pd.DataFrame(d)
        display(df)
        # for item in d.items():
        #    pprint(f"- {item}")
        # json.dumps(response, indent=1)
    else:
        pprint(f"Error in GET request (url) : {response.status_code}")

# Create the schema dump file

In [ ]:
if not FILE_OK:

    SCHEMA = "schema"
    STATISTICS = "statistics"
    TAGS = "tags"
#    BLOCKS = "blocks"
    CONESEARCH = "conesearch"
    OBJECTS = "objects"
    SOURCES = "sources"
    FP = "fp"
    CUTOUTS = "cutouts"


    SCHEMA_ENDPOINT = f"{API_V}{SCHEMA}"
    STATISTICS_ENDPOINT = f"{API_V}{STATISTICS}"
    TAGS_ENDPOINT = f"{API_V}{TAGS}"
#    BLOCKS_ENDPOINT = f"{API_V}{BLOCKS}"
    CONESEARCH_ENDPOINT =  f"{API_V}{CONESEARCH}"
    OBJECTS_ENDPOINT = f"{API_V}{OBJECTS}"
    SOURCES_ENDPOINT =  f"{API_V}{SOURCES}"
    FP_ENDPOINT = f"{API_V}{FP}"
    CUTOUTS_ENDPOINT = f"{API_V}{CUTOUTS}"
    
    #not used
    SCHEMA_API = f"{FINK_API}{SCHEMA_ENDPOINT}"
    STATISTICS_API = f"{FINK_API}{STATISTICS_ENDPOINT}"
    TAGS_API = f"{FINK_API}{TAGS_ENDPOINT}"
#    BLOCKS_API = f"{FINK_API}{BLOCKS_ENDPOINT}"
    CONESEARCH_API = f"{FINK_API}{CONESEARCH_ENDPOINT}"
    OBJECTS_API = f"{FINK_API}{OBJECTS_ENDPOINT}"
    SOURCES_API = f"{FINK_API}{SOURCES_ENDPOINT}"
    FP_API = f"{FINK_API}{FP_ENDPOINT}"
    CUTOUTS_API = f"{FINK_API}{CUTOUTS_ENDPOINT}"


    LIST_OF_STRUCT = [
        STATISTICS,
        TAGS,
    #    BLOCKS,
        CONESEARCH,
        OBJECTS,
        SOURCES,
        FP,
        CUTOUTS,
    ]
    
    LIST_OF_ENDPOINTS = [
        STATISTICS_ENDPOINT,
        TAGS_ENDPOINT,
        BLOCKS_ENDPOINT,
        CONESEARCH_ENDPOINT,
        OBJECTS_ENDPOINT,
        SOURCES_ENDPOINT,
        FP_ENDPOINT,
        CUTOUTS_ENDPOINT,
    ]
    
    #not used
    LIST_OF_URL = [
        STATISTICS_API,
        TAGS_API,
        BLOCKS_API,
        CONESEARCH_API,
        OBJECTS_API,
        SOURCES_API,
        FP_API,
        CUTOUTS_API,
    ]

    # extract schema of all endpooints
    all_schema = {}

    for struc in LIST_OF_STRUCT:
        print(">>>>> ", struc)
        df = result_from_post(SCHEMA_API, f"{API_V}{struc}")
        schema_dict = schema_to_dict(struc, df)
        
        for k, v in schema_dict.items():
            if k in all_schema:
                print(f"⚠️ Collision détectée: {k}")
            all_schema[k] = v

    # save
    import json
    from datetime import datetime
    # Génère la date au format YYYYMMDD
    date_du_jour = datetime.now().strftime('%Y%m%d')
    
    api_version = API_V.replace("/", "_").strip("_")
    fname = f"all_schema_{api_version}_{date_du_jour}.json"
    print("save in :",fname)
    with open(fname, "w") as f:
        json.dump(all_schema, f, indent=4)  # indent pour une meilleure lisibilité

# Some dump

In [ ]:
df = pd.DataFrame(list(all_schema.items()), columns=["endpoint/var", "doc"])

In [ ]:
from IPython.display import display, HTML


## Limite à N lignes AVANT de styliser
df_limited = df.head(10)

# Applique le style
styled_df = df_limited.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'normal',
    'word-wrap': 'break-word'
}).set_table_styles([{
    'selector': 'th',
    'props': [('text-align', 'center')]
}])

# Affiche le DataFrame stylisé et limité
display(HTML(styled_df.to_html()))